## チャットボットに天気説明機能も追加して更に便利にしよう！！！！！

・何ができるNotebookか、簡単な説明


　　以前作成したチャットボットの機能とwebチュートリアル2,6のシステムを流用し、APIでリアルタイムの市の天気を答えさせることができる。


・天気データの出典URL


　　Open-Meteo
　　[https://open-meteo.com/](https://open-meteo.com/)

　　地名から緯度・経度を検索するURL（2箇所に記載）
　　[https://geocoding-api.open-meteo.com/v1/search](https://geocoding-api.open-meteo.com/v1/search)?

　　緯度・経度から現在の天気・気温を取得するURL（1箇所に記載）
　　[https://api.open-meteo.com/v1/forecast](https://api.open-meteo.com/v1/forecast)?


・自分たちで追加した機能や特徴


　　地名から緯度経度を読み取り、天気urlに埋め込むシステム。
「市」などの末尾を付けるかつけないのかで判定が分かれるためfor関数で繰り返しの追加。"市"川"市"に対応させることに成功。


・これから改善したいこと、追加したい機


　　チャットボットの精度向上とより詳しいパターン分けをする。さらに返事をするための速度を上げるために効率的なコードに置き換える予定。


In [20]:
import json
from urllib.request import urlopen
from urllib.parse import urlencode#07で追加。テンプレート。

url = f"https://api.open-meteo.com/v1/forecast?latitude=35.86&longitude=139.97&current=temperature_2m&hourly=temperature_2m&forecast_days=1&timezone=Asia%2FTokyo"


import datetime  # 時間を扱うための標準機能
import json  # 前回から流用
import os  # 画像ファイルの存在確認を行う道具
import random
from urllib.parse import (
    quote,  # 日本語のキーワードをURL用に変換する道具
)
from urllib.request import Request, urlopen
from IPython.display import (
    Image,
    clear_output,
    display,
)  # 画面をクリアする道具・画像を表示する道具

# 辞書（dict）：Botのプロフィールを「名前: 値」の形で入れる
bot = {
    "name": "COIN",  # Botの名前
    "role": "ちょっとした検索、プログラミングの実験台",  # Botの役割
    "likes": ["検索", "勉強", "ゲーム"],  # リスト（list）：好きなもの三つ
}

# 気分ごとの顔。キーは normal / happy / surprised / tired の四つ
faces = {
    "normal": "🤖",
    "happy": "🤖✨",
    "surprised": "🤖❗",
    "tired": "🤖💤",
}

# 1. 判定用の言葉リスト
happy_words = [
    "元気",
    "楽しい",
    "うれしい",
    "嬉しい",
    "最高",
    "happy",
    "good",
    "great",
    "fun",
    "fine",
]
tired_words = [
    "元気じゃない",
    "つかれ",
    "疲れ",
    "眠い",
    "しんど",
    "つらい",
    "tired",
    "sleepy",
    "sad",
    "bad",
    "not good",
    "not happy",
]

# --- 挨拶用リストの切り分け ---
morning_words = ["おはよう", "オハヨウ", "おはよ"]
general_greetings = ["こんにちは", "こんばんは", "やあ", "ハロー", "はじめまして"]

# ★ 追加：天気判定用の単語リスト----------------------------------------------------------------------天気編で追加
now_word = ["今", "いま", "現在", "リアルタイム", "ナウ", "now"]
weather_q = ["天気", "てんき", "気温", "きおん", "空模様","天候","雨？","雨ですか？","雨だよね？"]

# --- コメント集 ---
morning_comments = [
    "今日も一日がんばろうね！☀️",
    "朝ごはんはしっかり食べた？☕",
    "今日も良い日になるといいね！✨",
]
daytime_comments = [
    "お昼ご飯は何にする？🍱",
    "午後も無理せず自分のペースでね！👍",
    "ちょっと一息ついて休憩も挟もうね ☕",
]
night_comments = [
    "今日も一日お疲れ様！ゆっくり休んでね 🌙",
    "美味しい夕飯食べた？🍲",
    "今日もお仕事や勉強、本当にお疲れ様！✨",
]

# 「おはよう」に添える時間帯別コメント
wake_up_daytime_comments = [
    "お昼頃に起きたんだね！よく眠れた？😊",
    "今日も良い一日になりますように！✨",
    "のんびり起きてリフレッシュできたかな？☕",
]
wake_up_night_comments = [
    "こんな時間に起きたの？お仕事や勉強お疲れ様！🌙",
    "夜起きたんだね！体調には気をつけてね",
    "静かな夜の時間、のんびり過ごそうね ✨",
]


# ガード付き画像表示関数（assetsフォルダ内の画像を読み込む）
def show_mood_image(current_mood):
    # assets フォルダ内の画像パスを指定 (.jpg / .png 両対応)
    jpg_path = os.path.join("assets", f"{current_mood}.jpg")
    png_path = os.path.join("assets", f"{current_mood}.png")

    # .jpg があればそれを使い、無ければ .png を探す
    target_path = (
        jpg_path
        if os.path.exists(jpg_path)
        else (png_path if os.path.exists(png_path) else None)
    )

    if target_path:
        try:
            display(Image(filename=target_path, width=180))
        except Exception:
            pass

# プロフィールを表示する
print(f"{faces['normal']} {bot['name']}｜{bot['role']}")

# 最初の画面表示用の初期値を用意
user_message = "（会話スタート）"
mood = "normal"
reply = f"こんにちは！{bot['name']}です。今日の気分はどう？（終わる時は「バイバイ」と入力してね）"

# 繰り返しの処理（whileループ）
while True:
    # 画面をリセットして最新の1画面だけにする
    clear_output(wait=True)

    # moodに応じた画像の表示
    show_mood_image(mood)

    # 見栄えを整える表示
    print("┌──────────────────────────────────────────┐")
    print(f"  👤 あなた：{user_message}")
    print("├──────────────────────────────────────────┤")
    print(f"  {faces.get(mood, '🤖')} {bot['name']}：{reply}")
    print("└──────────────────────────────────────────┘")

    # ユーザーの入力を受け取る
    try:
        user_message = input("💬 メッセージを入力：").strip()
    except KeyboardInterrupt:
        user_message = "バイバイ"

    # 終了判定
    end_words = [
        "バイバイ",
        "ばいばい",
        "bye",
        "exit",
        "終了",
        "end",
        "終わり",
        "おわり",
        "やめる",
        "ヤメる",
        "さようなら",
        "サヨウナラ",
        "またね",
        "おつかれ",
        "お疲れ",
        "オツカレ",
        "おやすみ",
        "オヤスミ",
        "ばい",
        "バイ",
        "ed",
        "えｄ","qwer",
    ]

    cleaned_input = (
        user_message.lower()
        .replace("「", "")
        .replace("」", "")
        .replace("『", "")
        .replace("』", "")
        .strip("。！!？?あ ")
    )

    if any(word in cleaned_input for word in end_words):
        clear_output(wait=True)
        show_mood_image("happy")
        print("┌──────────────────────────────────────────┐")
        print(
            f"  {faces['happy']} {bot['name']}：また、何時でも話しかけてくださいね～👋"
        )
        print("└──────────────────────────────────────────┘")
        break

    # 数当てゲーム（High & Low）ゾーン
    if "ゲーム" in user_message or "数当て" in user_message:
        target_number = random.randint(1, 100)
        bot["game_target"] = target_number
        mood = "happy"
        reply = "数当てゲームを始めましょう！1から100までの数字を入力して当ててみてください！（諦める時は「降参」と言ってね）"
        continue

    if "game_target" in bot:
        if user_message == "降参":
            ans = bot.pop("game_target")
            mood = "tired"
            reply = f"正解は【{ans}】でした！また挑戦してください！"
            continue

        if user_message.isdigit():
            num = int(user_message)
            ans = bot["game_target"]

            if num < ans:
                mood = "surprised"
                reply = f"【{num}】ですね！もっと「大きい（High）」数字です！"
            elif num > ans:
                mood = "surprised"
                reply = f"【{num}】ですか？もっと「小さい（Low）」数字です！"
            else:
                bot.pop("game_target")
                mood = "happy"
                reply = f"🎉 すごい！正解は【{ans}】でした！大正解！"
            continue

    # 自己紹介のためのゾーン
    you_words = [
        "あなた",
        "貴方",
        "貴女",
        "アナタ",
        "君",
        "きみ",
        "キミ",
        "お前",
        "おまえ",
        "オマエ",
        "あんた",
        "アンタ",
        "bot",
        "ボット",
        "ai",
        "AI",
        "Ai",
        "Bot",
        "BOT",
        "You",
        "お宅",
        "おたく",
        "御宅",
        "お主",
        "汝",
        "貴君",
        "貴殿",
        "貴公",
        "貴様",
        "キサマ",
        "きさま",
        "you",
        "ユー",
        "そなた",
        "ソナタ",
        "其方",
        "おぬし",
        "御主",
        "エーアイ",
        "こいん",
        "coin",
        "コイン",
        bot["name"].lower(),
    ]

    about_me = [
        "誰",
        "だれ",
        "何者",
        "ダレ",
        "なにもの",
        "何もの",
        "ナニモノ",
        "できる",
        "出来",
        "できま",
        "自己紹介",
        "機能",
        "特技",
        "使い方",
    ]
    about_like = ["好き", "すき", "趣味", "好物", "スキ"]
    about_creator_words = [
        "誰が作った",
        "作った人",
        "作者",
        "製作者",
        "生みの親",
        "誰が作",
        "開発者",
        "誰によって",
        "誰に",
        "誰の",
        "どなたに",
    ]
    about_age_words = [
        "何歳",
        "いくつ",
        "年齢",
        "誕生日",
        "生年月日",
        "いつ生まれた",
        "いつ作られた",
    ]
    about_status_words = [
        "元気",
        "調子",
        "なにしてる",
        "何してる",
        "忙しい",
        "起きてる","元気？",
    ]
    thank_words=["ありがと","サンク","感謝",]


    # 会話処理（挨拶）
    if any(word in user_message for word in morning_words):
        mood = "happy"
        current_hour = datetime.datetime.now().hour

        if 18 <= current_hour or current_hour < 5:
            comment = random.choice(wake_up_night_comments)
            if current_hour < 5:
                mood = "surprised"
        else:
            comment = random.choice(wake_up_daytime_comments)

        reply = f"おはようございます！{comment}"
        continue
    elif any(word in user_message for word in general_greetings):
        mood = "happy"
        current_hour = datetime.datetime.now().hour

        if 5 <= current_hour < 11:
            reply = f"おはようございます！{random.choice(morning_comments)}"
        elif 11 <= current_hour < 18:
            reply = f"こんにちは！{random.choice(daytime_comments)}"
        else:
            reply = f"こんばんは！{random.choice(night_comments)}"
        continue
    
              
    
    # ★ 追加：天気情報を取得するゾーン ----------------------------------------------------------------------## ここから課題のゾーン
     # 各都市の検索候補数
    match_count = 10

    # 過去何日分を含めるか
    past_days = 1

    # 今日を含む取得日数
    forecast_days = 3

    # 各都市の現地時間
    timezone = "auto"
    daily_fields = "precipitation_probability_max"

    # 0＝今日、1＝明日
    day_offset = 1



    if any(w in user_message for w in weather_q):#上記まではwikiのプログラムをコピー
        # カッコ「」や『』が含まれているか確認（Wiki機能と同じ判定）
        has_kakko = ("「" in user_message and "」" in user_message) or (
            "『" in user_message and "』" in user_message
        )

        if has_kakko:
            # カッコの中から都市名を取り出す（Wiki機能と全く同じ書き方）
            if "「" in user_message and "」" in user_message:
                city = user_message.split("「", 1)[1].split("」", 1)[0].strip()
            else:
                city = user_message.split("『", 1)[1].split("』", 1)[0].strip()

            if city:
                try: #都市ごとに「場所の検索→天気の取得→表示」を繰り返す   07のテンプレート。町の名前から経度を調べてくれる。「for city in cities:」は繰り返しで他の市も同時に調べるために使い物だったので必要なかった。

                    geo_url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode({
                                    "name": city, "count": match_count,
                                    "language": "ja", "countryCode": "JP"
                            })


                    with urlopen(geo_url, timeout=40) as response:
                        geo_data = json.load(response)
#後から追加○○市と○○で反応したりしないことがある。
                    if "results" not in geo_data:
                        # 試したい名前をそのまま並べる（足すパターン と 消すパターン）
                        # ① 後ろに「市区町村・都道府県」がついていたら、後ろ1文字を消した名前を作る
                        if city.endswith(("市", "区", "町", "村", "県", "府", "都")):#AIによるものもし「end（終わる）＋ with（〜で）」↓を実行
                            whpatterns = [city[:-1]]  # 更にここで市川市や四日市市の「市」がすべて消えるのを防ぐために、AIに聞いたsuffix（サフィックス）英語で「接尾辞（せつびじ）」だけを消す方法を使える。
                        else:
                            # ② ついていなければ、後ろに足した名前を作る
                            whpatterns = [
                                city + "市",
                                city + "区",
                                city + "県",
                                city + "府",
                                city + "町",
                                city + "村",
                            ]

                        # 上から1つずつ検索してみる。ループ関数最終的にはAIに聞いたが自分でも半分書けた。
                        for name in whpatterns:
                            geo_url = "https://geocoding-api.open-meteo.com/v1/search?" + urlencode({
                                "name": name, "count": match_count,
                                "language": "ja", "countryCode": "JP"
                            })
                            with urlopen(geo_url, timeout=40) as response:
                                geo_data = json.load(response)

                            # resultsが出たらOKで後ろに進むためのループ終了
                            if "results" in geo_data:
                                break


# 検索結果が存在する場合のみ処理を進める
                        # Wikiの if "extract" in data: と全く同じ書き方
                    if "results" in geo_data:
                        city_place = geo_data["results"][0]
                        
                        #week07のwebテンプレートから「見つかった緯度・経度を、天気APIに渡す」
                        lat = city_place["latitude"]
                        lon = city_place["longitude"]

                        
                        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,weather_code&timezone=Asia%2FTokyo"
                        headers = {"User-Agent": "Mozilla/5.0"}
                        req = Request(weather_url, headers=headers)
                        with urlopen(req, timeout=10) as response:
                            weather_data = json.load(response)
                        current_temp = weather_data["current"]["temperature_2m"]
                        weather_code = weather_data["current"]["weather_code"]#weather_code はOpen-Meteoがもともと用意している気象データ

                        # 天気コードを日本語に変換。AIにurlの読みかたを教えてもらった。weather_code はOpen-Meteoがもともと用意している気象データなので数値のみを変換する名称が必要。
                        weather_dict = {
                            0: "快晴",
                            1: "おおむね晴れ",
                            2: "一部曇り",
                            3: "曇り",
                            45: "霧",
                            48: "霧",
                            51: "霧雨",
                            53: "霧雨",
                            55: "強い霧雨",
                            56: "着氷性の霧雨",
                            57: "強い着氷性の霧雨",
                            61: "小雨",
                            63: "雨",
                            65: "大雨",
                            66: "着氷性の雨",
                            67: "強い着氷性の雨",
                            71: "小雪",
                            73: "雪",
                            75: "大雪",
                            77: "雪あられ",
                            80: "弱いにわか雨",
                            81: "にわか雨",
                            82: "激しいにわか雨",
                            85: "弱いにわか雪",
                            86: "激しいにわか雪",
                            95: "雷雨",
                            96: "ひょうを伴う雷雨",
                            99: "激しいひょうを伴う雷雨"
                        }

                        weather_text = weather_dict.get(weather_code, "")#回答するためにはweather_textをいう名称は絶対に必要。取り出す必要がある。

                        mood = "happy"#以降wikiプログラムからコピー
                        reply = f"現在の{city}の天気は【{weather_text}】、気温は【{current_temp}℃】です！"

                    else:
                        mood = "normal"
                        reply = f"申し訳ありません、{city} についての情報は見つかりませんでした…。"

                except Exception as e:
                        mood = "tired"
                        reply = "申し訳ありません、天気情報の取得中にエラーが発生いたしました。"

            else:
                            mood = "normal"
                            reply = "カッコの中が空っぽのようです！「『東京』の天気」のように、調べたい都市名を入れてみてくださいね！"
        else:
            mood = "normal"
            reply = "単語を正確に読み取るため、「『東京』の天気」のように「」や『』で囲んで話しかけてもらえると助かります！"



        continue



    # 言葉を説明のゾーン（Wikipedia検索）
    has_kakko = ("「" in user_message and "」" in user_message) or (
        "『" in user_message and "』" in user_message
    )
    search_hints = ["教え", "答え", "？", "?", "何", "とは"]
    has_hint = any(hint in user_message for hint in search_hints)

    if has_kakko:
        if "「" in user_message and "」" in user_message:
            clean_keyword = user_message.split("「", 1)[1].split("」", 1)[0].strip()
        else:
            clean_keyword = user_message.split("『", 1)[1].split("』", 1)[0].strip()

        if clean_keyword:
            try:
                encoded_keyword = quote(clean_keyword)
                url = f"https://ja.wikipedia.org/api/rest_v1/page/summary/{encoded_keyword}"
                headers = {"User-Agent": "Mozilla/5.0"}
                req = Request(url, headers=headers)

                with urlopen(req, timeout=100) as response:
                    data = json.load(response)

                if "extract" in data:
                    mood = "happy"
                    reply = f"【{clean_keyword} についてお調べしました！】\n{data['extract']}"
                else:
                    mood = "normal"
                    reply = f"申し訳ありません、{clean_keyword} についての情報は見つかりませんでした…。"
            except Exception:
                mood = "tired"
                reply = f"申し訳ありません、{clean_keyword} について検索処理中にエラーが発生いたしました。"
        else:
            mood = "normal"
            reply = "カッコの中が空っぽのようです！「『Python』について教えて」のように、調べたい言葉を入れてみてくださいね！"
        continue
    elif has_hint and not any(w in user_message for w in you_words):
        mood = "normal"
        reply = "単語を正確に読み取るため、「『Python』について教えて」のように「」や『』で囲んで話しかけてもらえると助かります！"
        continue


#自己紹介ゾーン改
    if (
        any(w in user_message for w in you_words)
        and any(w in user_message for w in about_age_words)
        and any(w in user_message for w in about_creator_words)
    ):
        mood = "happy"
        reply = "作者さんは『時間があったら入れるね』と言ったきり未実装なんです…。私は2026年10月25日に生まれました！"
        continue
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_creator_words
    ):
        mood = "happy"
        reply = "作者は「時間あったら入れとく～」っと言ってましたね。今は入ってないですね。これは...つめが甘い。"
        continue
    elif any(word in user_message for word in you_words) and any(
        word in user_message for word in about_me
    ):
        mood = "happy"
        reply = f"私は{bot['role']}の『{bot['name']}』と申します！🤖\n「『〇〇』について教えて」と話しかけてくれたらWikipediaで調べますね！"
        continue

    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_like
    ):
        mood = "happy"
        reply = "勉強とゲームが好きです！まだまだ知らない言葉も多いですが、頑張りますね。"
        continue
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_age_words
    ):
        mood = "happy"
        reply = "2026/10/25に生まれました！"
        continue
    elif any(w in user_message for w in you_words) and any(
        w in user_message for w in about_status_words
    ):
        mood = "happy"
        reply = "とっても元気ですよ！今日も一緒に頑張りましょう！"
        continue
    
    elif any(word in user_message for word in thank_words):
            mood = "happy"
            reply = "こちらこそ　ありがとうございます！"
            continue

    elif any(word in user_message for word in you_words):
            mood = "happy"
            reply = "はい！何でしょうか？お気軽にどうぞ！"
            continue




    # 気持ちのゾーン
    found_mood = False
    for word in happy_words:
        if word in user_message.lower():
            mood = "happy"
            reply = random.choice(
                [
                    "元気そうで何よりです！何か良いことありましたか？",
                    "その調子ですね！今日も楽しく過ごしていきましょう！",
                ]
            )
            found_mood = True
            break

    if not found_mood:
        for word in tired_words:
            if word in user_message.lower():
                mood = "tired"
                reply = random.choice(
                    [
                        "お疲れ様です。あまり無理はなさらないでくださいね ☕",
                        "今日はゆっくり休みましょう。温かい飲み物でも飲んでリフレッシュしてくださいね。",
                    ]
                )
                found_mood = True
                break

    if not found_mood:
        mood = "normal"
        reply = "すみません、上手く理解できませんでした。「『〇〇』について教えて」と聞くか、自己紹介を頼んでみてくださいね！"

┌──────────────────────────────────────────┐
  🤖✨ COIN：また、何時でも話しかけてくださいね～👋
└──────────────────────────────────────────┘
